In [2]:
import torch
from torch import nn
from torch.nn import functional as F

In [12]:
import math

class SelfAttention(nn.Module):
    mask = None
    @staticmethod
    def attention(q, k, v): # q, k, v -> (B, seq_len, head_size)
        head_size = q.shape[-1]
        attn_weight = (q @ k.transpose(-2, -1))/math.sqrt(head_size)
        if SelfAttention.mask != None:
            attn_weight.masked_fill_(SelfAttention.mask == 0, float('-inf'))
        attn_weight = F.softmax(attn_weight, dim=-1) # (B, seq_len, seq_len)
        return attn_weight @ v # (B, seq_len, seq_len) @ (B, seq_len, head_size) -> (B, seq_len, head_size)
    
    def __init__(self, d_model, head_size):
        super().__init__()
        self.query = nn.Linear(d_model, head_size)
        self.key = nn.Linear(d_model, head_size)
        self.value = nn.Linear(d_model, head_size)

    def forward(self, x): # (B, seq_len, d_model)
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        return SelfAttention.attention(q, k, v) # (B, seq_len, head_size)

def check_attention(B=10, seq_len=30, head_size=512):
    q = torch.randn(B, seq_len, head_size)
    k = torch.randn(B, seq_len, head_size)
    v = torch.randn(B, seq_len, head_size)
    SelfAttention.mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
    print("attention max diff", (F.scaled_dot_product_attention(q, k, v, SelfAttention.mask) - SelfAttention.attention(q, k, v)).abs().max().item())

check_attention()

attention max diff 3.3676624298095703e-06


In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "Number of attn heads must be divisible by d_model"
        head_size = d_model // num_heads
        self.num_heads = num_heads
        self.attn_heads = nn.ModuleList([SelfAttention(d_model, head_size) for _ in range(num_heads)])
        self.projection = nn.Linear(d_model, d_model)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.attn_heads], dim=-1)
        return self.projection(out)
    


def check_multihead_attention(B=10, seq_len=30, E=512, num_heads=128):
    assert E % num_heads == 0, "Embedding size must be divisible by num heads"
    nnma = nn.MultiheadAttention(embed_dim=E, num_heads=num_heads, batch_first=True)
    SelfAttention.mask = torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool))
    ma = MultiHeadAttention(d_model=E, num_heads=num_heads)
    W_q = nnma.in_proj_weight[:E, :].detach()
    W_k = nnma.in_proj_weight[E:E*2, :].detach()
    W_v = nnma.in_proj_weight[E*2:].detach()
    b_q = nnma.in_proj_bias[:E].detach()
    b_k = nnma.in_proj_bias[E:E*2].detach()
    b_v = nnma.in_proj_bias[E*2:].detach()
    W_o = nnma.out_proj.weight.detach()
    b_o = nnma.out_proj.bias.detach()
    head_size = E // num_heads
    start = 0
    for head in ma.attn_heads:
        head.query._parameters['weight'].data.copy_(W_q[start: start + head_size])
        head.query._parameters['bias'].data.copy_(b_q[start: start + head_size])
        head.key._parameters['weight'].data.copy_(W_k[start: start + head_size])
        head.key._parameters['bias'].data.copy_(b_k[start: start + head_size])
        head.value._parameters['weight'].data.copy_(W_v[start: start + head_size])
        head.value._parameters['bias'].data.copy_(b_v[start: start + head_size])
        start += head_size
    ma.projection.weight.data.copy_(W_o)
    ma.projection.bias.data.copy_(b_o)

    x = torch.randn(B, seq_len, E)
    print("Max diff multihead attention", (nnma(x, x, x, attn_mask=(SelfAttention.mask == False))[0].data - ma(x).data).abs().max().item())

check_multihead_attention()

Max diff multihead attention 0.0


In [85]:
print(x.shape)
q, k, v = (x @ nnma.in_proj_weight.detach().T + nnma.in_proj_bias.detach()).chunk(3, dim=-1)
q0 = x @ W_q.T + b_q
k0 = x @ W_k.T + b_k
v0 = x @ W_v.T + b_v
print("in_proj_weight deconstruction max error:", (q-q0).abs().max().item(), (k-k0).abs().max().item(), (v-v0).abs().max().item())

torch.Size([3, 2, 4])
in_proj_weight deconstruction max error: 0.0 0.0 0.0
